# MoP + DivPO — Kaggle Training Notebook

Trains 4 persona-specific LoRA adapters (SFT → DivPO) on Qwen2.5-0.5B-Instruct.
All outputs pushed to HF Hub after each persona — safe to resume after a session dies.

## Before running

1. **GPU**: Notebook settings → Accelerator → GPU T4 x2 (or P100)
2. **Internet**: Notebook settings → Internet → On
3. **Secret**: Add-ons → Secrets → Add → Name: `HF_TOKEN`, Value: your HuggingFace write token

## Pipeline

```
[DONE]   prepare_sft_datasets.py → SFT data on HF Hub
[Cell 4] train_sft.py            → SFT adapters pushed to HF Hub  (~1–2h per persona)
[Cell 6] prepare_divpo.py        → DivPO pairs generated + pushed  (~30–60 min per persona)
[Cell 8] train_divpo.py          → DivPO adapters pushed to HF Hub (~1h per persona)
```

Each phase runs **one persona at a time** with `--push` so progress survives the 12h session limit.

---
## Cell 1 — Install dependencies

> **After this cell completes, restart the kernel once** (Run → Restart kernel).
> Kaggle may have a stale `torchao` build that conflicts with PEFT LoRA injection.

In [ ]:
!pip install -q transformers peft trl accelerate bitsandbytes datasets huggingface_hub sentence-transformers
!pip uninstall -y -q torchao
print("Done. Restart the kernel now (Run → Restart kernel), then continue from Cell 2.")

---
## Cell 2 — Credentials + clone repo

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)

REPO_DIR = "/kaggle/working/mop-divpo-llm-counter-argument"

if not os.path.exists(REPO_DIR):
    !git clone https://github.com/DasonTio/mop-divpo-llm-counter-argument.git {REPO_DIR}
else:
    print("Repo already cloned — pulling latest")
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print("Working dir:", os.getcwd())

---
## Cell 3 — Verify GPU

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}:", torch.cuda.get_device_name(i))
else:
    print("No GPU — go to Notebook settings → Accelerator → T4 GPU")

---
## Phase 1 — SFT Training

Trains one LoRA adapter per persona on domain-specific SFT data.
Each adapter auto-pushed to `DasonTio/mop-divpo-coauthor/sft/{persona}/` after training.

Run one cell at a time to checkpoint between personas.
Estimated: **~1–2h per persona** on T4.

In [ ]:
# SFT — contrarian
!python scripts/train_sft.py --persona contrarian

In [ ]:
# SFT — systems_thinker
!python scripts/train_sft.py --persona systems_thinker

In [ ]:
# SFT — cross_domain_analogist
!python scripts/train_sft.py --persona cross_domain_analogist

In [ ]:
# SFT — minimalist
!python scripts/train_sft.py --persona minimalist

---
## Phase 2 — DivPO Dataset Generation

Generates preference pairs from the SFT adapters.
Each prompt → N candidate responses → `chosen` (rare+good) vs `rejected` (common).
`--push` uploads to HF Hub immediately after each persona — safe against session expiry.

Estimated: **~30–60 min per persona** on T4.

In [ ]:
# Download SFT JSONL for prompt pool
import os
from huggingface_hub import hf_hub_download

os.makedirs("data/processed/sft", exist_ok=True)
for persona in ["contrarian", "systems_thinker", "cross_domain_analogist", "minimalist"]:
    hf_hub_download(
        repo_id="DasonTio/mop-divpo-sft-data",
        filename=f"{persona}.jsonl",
        repo_type="dataset",
        local_dir="data/processed/sft",
        token=os.environ["HF_TOKEN"],
    )
    print(f"Downloaded {persona}.jsonl")

In [ ]:
# DivPO pairs — contrarian
!python scripts/prepare_divpo_datasets.py --persona contrarian --from-hub --candidate-count 4 --push

In [ ]:
# DivPO pairs — systems_thinker
!python scripts/prepare_divpo_datasets.py --persona systems_thinker --from-hub --candidate-count 4 --push

In [ ]:
# DivPO pairs — cross_domain_analogist
!python scripts/prepare_divpo_datasets.py --persona cross_domain_analogist --from-hub --candidate-count 4 --push

In [ ]:
# DivPO pairs — minimalist
!python scripts/prepare_divpo_datasets.py --persona minimalist --from-hub --candidate-count 4 --push

---
## Phase 3 — DivPO Training

Trains DPO on preference pairs, starting from SFT adapters.
Teaches the model to prefer rare-but-good responses over common ones.
Each adapter pushed to `DasonTio/mop-divpo-coauthor/divpo/{persona}/`.

Estimated: **~1h per persona** on T4.

In [ ]:
# DivPO training — contrarian
!python scripts/train_divpo.py --persona contrarian

In [ ]:
# DivPO training — systems_thinker
!python scripts/train_divpo.py --persona systems_thinker

In [ ]:
# DivPO training — cross_domain_analogist
!python scripts/train_divpo.py --persona cross_domain_analogist

In [ ]:
# DivPO training — minimalist
!python scripts/train_divpo.py --persona minimalist

---
## Verify — Load adapter and generate

In [ ]:
import torch
import sys
sys.path.insert(0, "src")

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_model = "Qwen/Qwen2.5-0.5B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(base_model)
base = AutoModelForCausalLM.from_pretrained(base_model, torch_dtype=torch.float16, device_map="auto")

model = PeftModel.from_pretrained(base, "DasonTio/mop-divpo-coauthor", subfolder="divpo/contrarian")
model.eval()

prompt = "Generate a counter-argument to this claim:\n\nRemote work is strictly better for productivity."
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=150, temperature=0.9, do_sample=True)
print(tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True))